In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

# Daten laden
X = pd.read_csv("x_values.csv").to_numpy(dtype=np.float32)
y = pd.read_csv("y_values.csv")["y"].to_numpy(dtype=np.int32)

num_features = X.shape[1]
num_classes = int(np.max(y) + 1)

# Split: 70/15/15 (stratifiziert)
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp
)

# Normalisierung NUR auf Train fitten
mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0) + 1e-8

X_train_n = (X_train - mu) / sigma
X_val_n   = (X_val   - mu) / sigma
X_test_n  = (X_test  - mu) / sigma

# für MCU speichern
np.savetxt("norm_mean.txt", mu)
np.savetxt("norm_std.txt", sigma)

# Modell (logits am Ende)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(num_features,)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(num_classes)
])

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

# Training
history = model.fit(
    X_train_n, y_train,
    validation_data=(X_val_n, y_val),
    epochs=80,
    batch_size=16,
    verbose=1
)

# Test
logits = model.predict(X_test_n)
y_pred = np.argmax(logits, axis=1)

test_acc = float(np.mean(y_pred == y_test))
print("Test accuracy:", test_acc)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


Epoch 1/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 116ms/step - accuracy: 0.0000e+00 - loss: 1.1758 - val_accuracy: 0.2857 - val_loss: 1.1001
Epoch 2/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.2036 - loss: 1.1220 - val_accuracy: 0.5714 - val_loss: 1.0572
Epoch 3/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.4315 - loss: 1.0666 - val_accuracy: 0.7143 - val_loss: 1.0196
Epoch 4/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.6898 - loss: 1.0189 - val_accuracy: 0.7143 - val_loss: 0.9878
Epoch 5/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7132 - loss: 0.9688 - val_accuracy: 0.7143 - val_loss: 0.9576
Epoch 6/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6820 - loss: 0.9549 - val_accuracy: 0.7143 - val_loss: 0.9294
Epoch 7/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6664 - loss: 0.9225 - val_accuracy: 0.7143 - val_loss: 0.9049
Epoch 8/80
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6664 - loss: 0.8994 - val_accuracy: 0.8571 - val_loss: 0.

In [6]:
import tensorflow as tf

# 1) Softmax-Modell für Wahrscheinlichkeiten
probability_model = tf.keras.Sequential([model, tf.keras.layers.Softmax()])

# 2) TFLite Export
converter = tf.lite.TFLiteConverter.from_keras_model(probability_model)
tflite_model = converter.convert()

with open("device_model.tflite", "wb") as f:
    f.write(tflite_model)

print("✅ Saved: device_model.tflite")
print("Input features:", 7, "Output classes:", 3)


Saved artifact at '/tmp/tmpqpp57552'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 7), dtype=tf.float32, name='keras_tensor_28')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  133549998127824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133549998128592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133549998127056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133549998118992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133549998127632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133549998118608: TensorSpec(shape=(), dtype=tf.resource, name=None)
✅ Saved: device_model.tflite
Input features: 7 Output classes: 3
